In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import joblib

# import some libraries for training
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import StratifiedKFold, cross_val_score
import torch.nn as nn
import torch.optim as optim
import xgboost as xgb
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error
import matplotlib.pyplot as plt

from helper_modules import phase2_preprocess_data, perform_one_hot_encode

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

# disable some warnings
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="hyperopt")


In [ ]:
df = pd.read_csv(os.path.join('datasets', 'flextrack_phase1_train.csv'))
print(df.shape)
display(df.head())

In [ ]:
# Try extracting data that has Demand_Response_Flag = 0 and Demand_Response_Capacity_kW != 0.0
df1 = df[
    (df['Demand_Response_Flag']==0) &
    (df['Demand_Response_Capacity_kW'] != 0.0)
].copy(deep=True)
print (df1.shape[0])

df1 = df[
    df['Demand_Response_Flag']==0
].copy(deep=True)
df2 = df1[
    df1['Demand_Response_Capacity_kW'] != 0.0
].copy(deep=True)
print (df2.shape[0])
# We observe that if Demand_Response_Flag = 0, then Demand_Response_Capacity_kW is always 0.0

In [ ]:
# perform some preprocessing
df = phase2_preprocess_data(df)
df = perform_one_hot_encode(df, col_name='Demand_Response_Flag')
display(df.head())

# Visualize the distribution of Demand_Response_Capacity_kW
fig, (ax1, ax2) = plt.subplots(1,2,figsize=(9, 3))
df['Demand_Response_Capacity_kW'].hist(bins=100, ax=ax1)
ax1.set_title('Distribution of DRC(kW) w/ zeros')
ax1.set_xlabel('Demand_Response_Capacity_kW')
ax1.set_ylabel('Count')

df[
    df['Demand_Response_Capacity_kW'] != 0.0
]['Demand_Response_Capacity_kW'].hist(bins=100, ax=ax2)
ax2.set_title('Distribution of DRC(kW) w/out zeros')
ax2.set_xlabel('Demand_Response_Capacity_kW')
ax2.set_ylabel('Count')
plt.show()

In [ ]:
# suspects.remove('Building_Power_kW')
df = df.drop_duplicates()
X = df.drop(columns=['Demand_Response_Capacity_kW'])
y = df['Demand_Response_Capacity_kW']


def train_two_part_xgb(X: pd.DataFrame, y: pd.Series, random_state=42):
   # 0) Make zero-indicator for stratification & classification
   y_is_nonzero = (y != 0).astype(int)
   # 1) Train/valid split (stratify on zero vs nonzero)
   X_tr, X_te, y_tr, y_te, nz_tr, nz_te = train_test_split(
       X, y, y_is_nonzero, test_size=0.01, random_state=random_state, stratify=y_is_nonzero
   )
   # (Optional) scale numeric features if distributions are wildly different.
   # Keep it simple: scale everything numeric; tree models don't need it but it can help if you later swap models.
   X_tr_scaled = X_tr.copy()
   X_te_scaled = X_te.copy()
   # 2) Classifier: P(nonzero | x)
   pos = nz_tr.sum()
   neg = len(nz_tr) - pos
   scale_pos_weight = (neg / max(pos, 1)) if pos > 0 else 1.0
   clf = xgb.XGBClassifier(
       n_estimators=2000,
       max_depth=5,
       learning_rate=0.05,
       subsample=0.8,
       colsample_bytree=0.8,
       reg_lambda=5.0,
       reg_alpha=0.5,
       min_child_weight=10.0,
       objective='binary:logistic',
       eval_metric='logloss',
       tree_method='hist',
       random_state=random_state,
       scale_pos_weight=scale_pos_weight
   )
   clf.fit(
       X_tr_scaled, nz_tr,
       eval_set=[(X_te_scaled, nz_te)],
       verbose=False,
   )
   # 3) Regressor on non-zeros: E[y | nonzero, x]
   mask_tr_nz = (y_tr != 0)
   mask_te_nz = (y_te != 0)
   # Use MAE objective (robust to outliers) – good default for messy tails
   reg = xgb.XGBRegressor(
       n_estimators=2000,
       max_depth=6,
       learning_rate=0.05,
       subsample=0.8,
       colsample_bytree=0.9,
       reg_lambda=1.0,
       reg_alpha=0.0,
       min_child_weight=2.0,
       objective='reg:absoluteerror',  # a.k.a. MAE loss
       eval_metric='mae',
       tree_method='hist',
       random_state=random_state
   )
   reg.fit(
       X_tr_scaled[mask_tr_nz], y_tr[mask_tr_nz],
       eval_set=[(X_te_scaled[mask_te_nz], y_te[mask_te_nz])],
       verbose=False,
   )
   # 4) Inference: E[y] = P(nonzero|x)*E[y|nonzero,x]
   p_nonzero_te = clf.predict_proba(X_te_scaled)[:, 1]
   y_hat_cond_te = reg.predict(X_te_scaled)
   y_hat_te = p_nonzero_te * y_hat_cond_te
   # 5) Metrics
   rmse = np.sqrt(mean_squared_error(y_te, y_hat_te))
   mae = mean_absolute_error(y_te, y_hat_te)
   # extra diagnostics
   # classification quality on zero vs non-zero
   auc = roc_auc_score(nz_te, p_nonzero_te)
   # error on the hard part (non-zeros only)
   nz_mae = mean_absolute_error(y_te[mask_te_nz], y_hat_te[mask_te_nz]) if mask_te_nz.any() else np.nan
   nz_rmse = np.sqrt(mean_squared_error(y_te[mask_te_nz], y_hat_te[mask_te_nz])) if mask_te_nz.any() else np.nan
   results = {
       "Overall_MAE": mae,
       "Overall_RMSE": rmse,
       "Nonzero_MAE": nz_mae,
       "Nonzero_RMSE": nz_rmse,
       "ZeroRate_Test": 1 - nz_te.mean(),
       "Classifier_AUC": auc,
       'y_hat_te' : y_hat_te,
         'y_te' : y_te
   }
   return clf, reg, results

clf, reg, results = train_two_part_xgb(X,y)
print(
   results['Overall_MAE'],
   results['Overall_RMSE'],
   results['Nonzero_MAE'],
   results['Nonzero_RMSE'],
   results['ZeroRate_Test'],
   results['Classifier_AUC']
)

In [ ]:
# Save models
joblib.dump(clf, './models/phase2_xgb_clf_nz_model.pkl')
joblib.dump(reg, './models/phase2_xgb_reg_nz_model.pkl')
print("Models saved.")

In [ ]:
y_pred = results['y_hat_te']
y_true = results['y_te']

plt.figure(figsize=(10, 5))
plt.hist(y_true, bins=100, alpha=0.5, label='y_true')
plt.hist(y_pred, bins=100, alpha=0.5, label='y_pred')
plt.legend()
plt.xlabel('Demand_Response_Capacity_kW')
plt.ylabel('Count')
plt.title('Distribution of y_true and y_pred')
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
y_true_nonzero = y_true[y_true != 0.0]
y_pred_nonzero = y_pred[y_true != 0.0]
plt.hist(y_true_nonzero, bins=100, alpha=0.7, color='tab:blue')
plt.hist(y_pred_nonzero, bins=100, alpha=0.7, color='tab:red')
plt.xlabel('Demand_Response_Capacity_KW')
plt.ylabel('Count')
plt.title('Distribution of y_true (non-zero values only)')
plt.show()

In [ ]:
np.min(y_pred), np.max(y_pred), np.mean(y_pred), np.std(y_pred)

In [ ]:
X.head()